# Exploring Stock Data using brapi-dev api

## Load ingested data into spark tables

In [1]:
from spark import loader

spark = loader.create_spark_session("brapi-api")

loader.load_spark_tables(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 18:35:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/06 18:35:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Fundamental Analysis Metrics

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

### Companies with Consistent Profits

In [3]:
# Create a window to grab the most recent 4 quarters per stock
window_spec = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

consistent_profits_df = (
    spark.table("incomestatementhistoryquarterly")
    # Rank quarters by date
    .withColumn("rank", F.row_number().over(window_spec))
    # Filter for the last year of data (last 4 quarters)
    .filter(F.col("rank") <= 4)
    # Check if the company was profitable in each quarter (1 if true, 0 if false)
    .withColumn("is_profitable", F.when(F.col("netIncome") > 0, 1).otherwise(0))
    # Aggregate by stock to see how many of the 4 quarters were profitable
    .groupBy("stock")
    .agg(
        F.sum("is_profitable").alias("profitable_quarters_count"),
        F.avg("netIncome").alias("avg_quarterly_net_income")
    )
    # Target companies that hit 4 out of 4 profitable quarters
    .filter(F.col("profitable_quarters_count") == 4)
    .select("stock", "avg_quarterly_net_income")
)

consistent_profits_df.toPandas().head()

,stock,avg_quarterly_net_income
0,PETR4,2.700875e+10


### Companies with Low Debt in Proportion to EBITDA

In [4]:
# Window spec to get the absolute latest recorded quarter
latest_window = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

low_debt_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    # Grab the most recent quarter
    .filter(F.col("rank") == 1)
    # Avoid division by zero or working with companies with negative EBITDA
    .filter(F.col("ebitda") > 0)
    # Annualizing quarterly EBITDA (EBITDA * 4) to accurately measure Debt/EBITDA
    .withColumn("annualized_ebitda", F.col("ebitda") * 4)
    .withColumn("debt_to_ebitda", F.col("totalDebt") / F.col("annualized_ebitda"))
    # Filter for companies with a conservative ratio (e.g., Debt/EBITDA < 2.0)
    .filter(F.col("debt_to_ebitda") < 2.0)
    .select("stock", "totalDebt", "ebitda", "debt_to_ebitda")
    .orderBy("debt_to_ebitda")
)

low_debt_df.toPandas().head()

,stock,totalDebt,ebitda,debt_to_ebitda
0,PETR4,676977000000,230884000000,0.733027
1,MGLU3,10525008000,3127691000,0.841276
2,VALE3,194700000000,51387000000,0.947224


### Companies with Consistency Growth

In [5]:
# Window spec to pull the last 4 quarters
growth_window = Window.partitionBy("stock").orderBy(F.col("endDate").desc())

consistent_growth_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(growth_window))
    .filter(F.col("rank") <= 4)
    # A quarter is marked '1' if growth was positive (e.g., > 5% YoY growth)
    .withColumn("is_growing", F.when(F.col("revenueGrowth") > 0.05, 1).otherwise(0))
    .groupBy("stock")
    .agg(
        F.sum("is_growing").alias("consistent_growth_quarters"),
        F.avg("revenueGrowth").alias("avg_revenue_growth")
    )
    # Keep companies that hit our target growth benchmark in all 4 quarters
    .filter(F.col("consistent_growth_quarters") == 4)
    .select("stock", "avg_revenue_growth")
    .orderBy(F.col("avg_revenue_growth").desc())
)

consistent_growth_df.toPandas().head()

,stock,avg_revenue_growth
0,ITUB4,0.155728


### Return on Equity (ROE) & Return on Assets (ROA)

In [8]:
# Ensure you are checking the latest quarter's efficiency
efficiency_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    # Filters out highly distorted or negative equity anomalies
    .filter((F.col("returnOnEquity") > 0.15) & (F.col("returnOnAssets") > 0.05))
    .select("stock", "returnOnEquity", "returnOnAssets")
)

efficiency_df.toPandas().head()

,stock,returnOnEquity,returnOnAssets
0,PETR4,0.242672,0.086701


### Free Cash Flow (FCF) Margin

In [11]:
fcf_health_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    .filter(F.col("totalRevenue") > 0)
    # FCF Margin = Free Cash Flow / Total Revenue
    .withColumn("fcf_margin", F.col("freeCashflow") / F.col("totalRevenue"))
    .filter(F.col("fcf_margin") > 0.10)
    .select("stock", "freeCashflow", "fcf_margin")
)

fcf_health_df.toPandas().head()

,stock,freeCashflow,fcf_margin
0,ITUB4,92871000000,0.238944
1,MGLU3,9124675000,0.236881
2,PETR4,80740000000,0.162099


### PEG Ratio (Price/Earnings-to-Growth)

In [9]:
growth_valuation_df = (
    spark.table("defaultkeystatisticshistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    # Look for companies whose stock price isn't outrunning their earnings speed
    .filter((F.col("pegRatio") > 0) & (F.col("pegRatio") <= 1.0))
    .select("stock", "pegRatio", "trailingPE")
)

growth_valuation_df.toPandas().head()

,stock,pegRatio,trailingPE
0,ITUB4,0.950632,9.567522
1,PETR4,0.045240,5.504934


### Short-Term Liquidity: The Current Ratio

In [12]:
liquidity_df = (
    spark.table("financialdatahistoryquarterly")
    .withColumn("rank", F.row_number().over(latest_window))
    .filter(F.col("rank") == 1)
    .filter(F.col("currentRatio") >= 1.5)
    .select("stock", "currentRatio", "quickRatio")
)

liquidity_df.toPandas().head()

,stock,currentRatio,quickRatio


### Good Stock Opportunities

In [13]:
stock_opportunities_df = (
    # Baseline active tickers
    spark.table("activetickers").select("stock", "name", "sector")
    # 1. Profits & Growth checks from previous queries
    .join(consistent_profits_df, on="stock", how="inner")
    .join(consistent_growth_df, on="stock", how="inner")
    # 2. Add Long-term Leverage safety
    .join(low_debt_df.select("stock", "debt_to_ebitda"), on="stock", how="inner")
    # 3. Add short-term liquidity safety
    .join(liquidity_df, on="stock", how="inner")
    # 4. Add Capital Efficiency 
    .join(efficiency_df, on="stock", how="inner")
    # 5. Add FCF Quality Check
    .join(fcf_health_df, on="stock", how="inner")
    # 6. Add Growth-Adjusted Valuation
    .join(growth_valuation_df, on="stock", how="inner")
)

stock_opportunities_df.toPandas().head()

,stock,name,sector,avg_quarterly_net_income,avg_revenue_growth,debt_to_ebitda,currentRatio,quickRatio,returnOnEquity,returnOnAssets,freeCashflow,fcf_margin,pegRatio,trailingPE


## Full Data Catalog

In [7]:
for table in spark.catalog.listTables():
    print(f"---- {table.name} ----\n")
    spark.table(table.name).printSchema()

---- activetickers ----

root
 |-- change: double (nullable = true)
 |-- close: double (nullable = true)
 |-- logo: string (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- name: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- stock: string (nullable = true)
 |-- subType: string (nullable = true)
 |-- type: string (nullable = true)
 |-- volume: long (nullable = true)

---- balancesheethistoryquarterly ----

root
 |-- stock: string (nullable = true)
 |-- type: string (nullable = true)
 |-- endDate: string (nullable = true)
 |-- cash: long (nullable = true)
 |-- shortTermInvestments: long (nullable = true)
 |-- netReceivables: long (nullable = true)
 |-- inventory: long (nullable = true)
 |-- otherCurrentAssets: long (nullable = true)
 |-- totalCurrentAssets: long (nullable = true)
 |-- longTermInvestments: long (nullable = true)
 |-- propertyPlantEquipment: long (nullable = true)
 |-- otherAssets: long (nullable = true)
 |-- totalAssets: long (nullable